# Historic Data Comparison

The next cell is almost equal to the above but we compare the recently retrieved dataset against a historical one.

A historical sample from 2021 can be downloaded [here](https://8bit-inferno.de/_persistent/save_120k_dc_all_2021.pickle).

---

**Safe to continue from here if you can provide 'save_120k_dc_all_2021.pickle'** and have run the cell above.


In [59]:
import numpy as np
import pandas as pd

import os
import sys
from datetime import datetime  # for time measurement
import requests
import urllib

import pickle

In [60]:
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
#
# If 'loadMissingData' is set to True, sample files will be downloaded automatically
# This is a RECOMMENDED SETTING as it will take care of a some extra work for you
#
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
loadMissingData = True


recordDatabaseFile = "./save_120k_dc_all_DEMO_2026.pickle"
recordDatabaseFileLarge = "./save_246k_dc_all_DEMO.pickle"

In [61]:
# ANSI-based styling
# Standard foreground colors (30–37)


BLACK = "\033[30m"
RED = "\033[31m"
GREEN = "\033[32m"  # baseline
YELLOW = "\033[33m"
BLUE = "\033[34m"
MAGENTA = "\033[35m"
CYAN = "\033[36m"
WHITE = "\033[37m"

# Bright variants (90–97)
BRIGHT_GREEN = "\033[92m"
BRIGHT_RED = "\033[91m"

# Styles
BOLD = "\033[1m"
DIM = "\033[2m"
UNDERLINE = "\033[4m"
RESET = "\033[0m"


# utility functions
# print a line with the current timestamp
def printLog(text):
    now = str(datetime.now())
    print("[" + now + "]\t" + text)
    # forces to output the result of the print command immediately, see: http://stackoverflow.com/questions/230751/how-to-flush-output-of-python-print
    sys.stdout.flush()


# load a file from an URL and save it to a folder
def saveFileFromURL(file_url, output_dir):
    # get the path part of the URL
    path = urllib.parse.urlparse(file_url).path
    # remove the 8bit-inferno.de-specific path part
    file_part = path.replace("/_persistent/", "")

    printLog(f"Downloading {file_url} to '{output_dir}/{file_part}'")

    # send a GET request to the URL
    response = requests.get(file_url)
    # check if the request was successful
    if response.status_code == 200:
        # Open a local file in binary write mode
        with open(output_dir + "/" + file_part, "wb") as file:
            # Write the content of the response to the file
            file.write(response.content)
        printLog("Finished download.")
        return True
    else:
        printLog(
            f"Failed to download file with HTTP status code: {response.status_code}"
        )
        return False

## Get The Current Dataset


In [62]:
if loadMissingData:
    if not os.path.isfile(recordDatabaseFile):
        saveFileFromURL(
            "https://8bit-inferno.de/_persistent/save_120k_dc_all_DEMO_2026.pickle",
            ".",
        )

# load the records
# uncomment the next 3 lines if you are continuing the execution of this notebook at a later point in time
# printLog("Loading pickled records...")
savedRecords = pickle.load(open(recordDatabaseFile, "rb"))
# printLog("Finished loading pickled records.")

availableKeys = dict()

# check for all keys present in the previously downloaded dataset
for i, r in enumerate(savedRecords):
    for k in r.keys():
        if not k in availableKeys:
            availableKeys[k] = 1
        else:
            availableKeys[k] = availableKeys[k] + 1

print(availableKeys)

# create a dictionary for the records
values = dict()
# take the keys as they have found within the downloaded OAI records
keys = availableKeys.keys()
# for every metadata field, create an empty array as the content of the dictionary filed under the key 'k'
for k in keys:
    values[k] = []
# in addition, store the PPN (the SBB's unique identifier for digitized content)
# values["PPN"]=[]

# iterate over all saved records
# note: in 2026 this creation of a dataframe has become pretty verbose, for a more straightforward approach see the cells below
# however, we keep this ancient code here as it shows detailed how it was once done, even when Python 2 was still a thing
for record in savedRecords:
    # we cannot iterate over the keys of record.metadata directly because not all records cotain the same fields,...
    for k in keys:
        # thus we check if the metadata field 'k' has been created above
        if k in values:
            # append the metadata fields to the dictionary created above
            # if the metadata field 'k' is not available input "None" instead
            # values[k].append(record.get(k,["None"])[0].encode('ISO-8859-1'))
            if k in record:
                value = record.get(k)[0]
                if value:
                    if value.isdigit():
                        value = int(value)
                    else:
                        # p27 value=value.encode('ISO-8859-1')
                        # value=value.encode('ISO-8859-1').decode("utf-8", "backslashreplace")
                        value = value
                    values[k].append(value)
                else:
                    values[k].append(np.nan)
            else:
                values[k].append(np.nan)
# create a data frame from the
# p27 df=pd.DataFrame(pd.to_numeric(values,errors='coerce'))
df = pd.DataFrame(values)
# df['date']=pd.to_numeric(df['date'],errors='ignore',downcast='integer')
# df=pd.DataFrame(values)
# df=df.convert_objects(convert_dates=False, convert_numeric=True, convert_timedeltas=False, copy=True)
df.shape

{'title': 120000, 'creator': 79390, 'subject': 119694, 'publisher': 39511, 'type': 120000, 'format': 120000, 'identifier': 120000, 'language': 73739, 'date': 117969, 'coverage': 100947, 'rights': 117045, 'PPN': 120000}


(120000, 12)

In [63]:
df

,title,creator,subject,publisher,type,format,identifier,language,date,coverage,rights,PPN
0,"Collegium Ethicum, In Quo De Summo Hominis Bon...","Liebenthal, Christianus",Historische Drucke,Hampelius,monograph,application/mets+xml,PPN1000000974,lat,1620,Giessae Hessorum,Public Domain Mark 1.0,PPN1000000974
1,"Augustini Niphi, Philosophi Suessani De Auguri...","Niphus, Augustinus",Historische Drucke,Egenolphus,monograph,application/mets+xml,PPN100000127X,lat,1614,Marpurgi,Public Domain Mark 1.0,PPN100000127X
2,Concerto pour la Flûte avec Accompagnement de ...,"Danzi, Franz",Musiknoten,Breitkopf & Härtel,monograph,application/mets+xml,PPN1000006808,NaN,1806,Leipsic,CC BY-NC-SA 4.0 International,PPN1000006808
3,Schuldige Condolentz-Zeilen Womit seinem Gelie...,"Koltemann, Otto Benedict",Historische Drucke,Königliche privil. Buchdruckerey,monograph,application/mets+xml,PPN1000056597,ger,1719,Glückstadt,Public Domain Mark 1.0,PPN1000056597
4,Johann Friedrich Böttchers der Arzney und Wund...,"Böttcher, Johann Friedrich",Historische Drucke,Oehmigke,monograph,application/mets+xml,PPN1000059669,ger,1796,Berlin,Public Domain Mark 1.0,PPN1000059669
...,...,...,...,...,...,...,...,...,...,...,...,...
119995,"Programma In Honorem Spiritus Sancti, Die Pent...",NaN,Historische Drucke,Typis Reusnerianis,monograph,application/mets+xml,PPN625200527,lat,1692,Regiomonti,Public Domain Mark 1.0,PPN625200527
119996,Kurtze Nachricht/ Wie es Bey der Gott gebe frö...,"Büttner, Johan Henrich",Historische Drucke,Stern,monograph,application/mets+xml,PPN625216415,ger,1708,Lüneburg,Public Domain Mark 1.0,PPN625216415
119997,Dissertation Sur L'Article Du Symbole Des Apot...,"Auret La Grave, David",Historische Drucke,Michaelis,monograph,application/mets+xml,PPN625218329,fre,1725,Berlin,Public Domain Mark 1.0,PPN625218329
119998,"Programma In Honorem Jesu Christi, a David, In...",NaN,Historische Drucke,Typis Reusnerianis,monograph,application/mets+xml,PPN625225414,lat,1692,Regiomonti,Public Domain Mark 1.0,PPN625225414


## Get Historic Datasets


In [64]:
# load the records
printLog("Loading pickled records...")
# indicates whether a sample file could be found (important for later comparisons)
oldPickleAvailable = False

availableYears = [2019, 2021, 2022]
seen_PPNs_over_years = []  # holds all PPNs seen in each year
seen_PPNs = {k: [] for k in availableYears}

df_per_year = dict()

for year in availableYears:
    oldPickle = f"save_120k_dc_all_{year}_DEMO.pickle"
    if not os.path.isfile(oldPickle):
        printLog("The historical sample file could not be found.")
        # check if sample data is available and load it if automatic downloads are set by the user
        if loadMissingData:
            oldPickleAvailable = saveFileFromURL(
                f"https://8bit-inferno.de/_persistent/save_120k_dc_all_{year}_DEMO.pickle",
                ".",
            )
            oldPickle = f"save_120k_dc_all_{year}_DEMO.pickle"

    if os.path.isfile(oldPickle) or oldPickleAvailable:
        # uncomment this line if you are continuing the execution of this notebook at a later point in time
        savedRecords = pickle.load(open(oldPickle, "rb"))
        printLog(f"Finished loading pickled records for year {year}.")

        availableKeys = dict()

        # check for all keys present in the previously downloaded dataset
        for i, r in enumerate(savedRecords):
            for k in r.keys():
                if not k in availableKeys:
                    availableKeys[k] = 1
                else:
                    availableKeys[k] = availableKeys[k] + 1

        print(availableKeys)

        # create a dictionary for the records
        values = dict()
        # take the keys as they have found within the downloaded OAI records
        keys = availableKeys.keys()
        # for every metadata field, create an empty array as the content of the dictionary filed under the key 'k'
        for k in keys:
            values[k] = []
        # in addition, store the PPN (the SBB's unique identifier for digitized content)
        # values["PPN"]=[]

        # iterate over all saved records
        # note: in 2026 this creation of a dataframe has become pretty verbose, for a more straightforward approach see the cells below
        # however, we keep this ancient code here as it shows detailed how it was once done, even when Python 2 was still a thing
        for record in savedRecords:
            ppn = record["PPN"][0]
            seen_PPNs_over_years.append(ppn)
            seen_PPNs[year].append(ppn)
            # we cannot iterate over the keys of record.metadata directly because not all records cotain the same fields,...
            for k in keys:
                # thus we check if the metadata field 'k' has been created above
                if k in values:
                    # append the metadata fields to the dictionary created above
                    # if the metadata field 'k' is not available input "None" instead
                    # values[k].append(record.get(k,["None"])[0].encode('ISO-8859-1'))
                    if k in record:
                        value = record.get(k)[0]
                        if value:
                            if value.isdigit():
                                value = int(value)
                            else:
                                # p27 value=value.encode('ISO-8859-1')
                                # value=value.encode('ISO-8859-1').decode("utf-8", "backslashreplace")
                                value = value
                            values[k].append(value)
                        else:
                            values[k].append(np.nan)
                    else:
                        values[k].append(np.nan)
        # create a data frame from the
        # p27 df=pd.DataFrame(pd.to_numeric(values,errors='coerce'))
        df_hist = pd.DataFrame(values)
        # df_2021['date']=pd.to_numeric(df_2021['date'],errors='ignore',downcast='integer')
        # df=pd.DataFrame(values)
        # df=df.convert_objects(convert_dates=False, convert_numeric=True, convert_timedeltas=False, copy=True)
        df_hist.shape
        df_per_year[year] = df_hist
        oldPickleAvailable = True

[2026-11-03 20:40:32.645746]	Loading pickled records...
[2026-11-03 20:40:33.169853]	Finished loading pickled records for year 2019.
{'title': 120000, 'creator': 95345, 'subject': 119991, 'type': 120000, 'format': 120000, 'identifier': 120000, 'language': 103710, 'date': 120000, 'relation': 115938, 'coverage': 98717, 'PPN': 120000, 'publisher': 63421, 'rights': 76008}
[2026-11-03 20:40:34.090439]	Finished loading pickled records for year 2021.
{'title': 120000, 'creator': 96090, 'subject': 119989, 'type': 120000, 'format': 120000, 'identifier': 120000, 'language': 89713, 'date': 120000, 'relation': 114947, 'coverage': 101010, 'PPN': 120000, 'publisher': 55127, 'rights': 90158}
[2026-11-03 20:40:34.979173]	Finished loading pickled records for year 2022.
{'title': 120000, 'creator': 88841, 'subject': 119901, 'publisher': 58920, 'type': 120000, 'format': 120000, 'identifier': 120000, 'language': 73020, 'date': 117074, 'coverage': 109411, 'rights': 113648, 'PPN': 120000}


In [65]:
df_per_year2 = dict()
uniquePPNs_over_years = dict()  # holds all PPNs seen in each year

for year in availableYears:
    oldPickle = f"save_120k_dc_all_{year}_DEMO.pickle"
    if not os.path.isfile(oldPickle):
        printLog("The historical sample file could not be found.")
        # check if sample data is available and load it if automatic downloads are set by the user
        if loadMissingData:
            oldPickleAvailable = saveFileFromURL(
                f"https://8bit-inferno.de/_persistent/save_120k_dc_all_{year}_DEMO.pickle",
                ".",
            )
            oldPickle = f"save_120k_dc_all_{year}_DEMO.pickle"

    if os.path.isfile(oldPickle) or oldPickleAvailable:
        # uncomment this line if you are continuing the execution of this notebook at a later point in time
        savedRecords = pickle.load(open(oldPickle, "rb"))
        printLog(f"Finished loading pickled records for year {year}.")

        # as the values of the dict are lists, we have to make sure to the first element of each list to create a proper dataframe
        r = [
            {key: values[0] for key, values in record.items()}
            for record in savedRecords
        ]
        df_per_year2[year] = pd.DataFrame(r)
        uniquePPNs = set(df_per_year2[year]["PPN"])
        uniquePPNs_over_years[year] = uniquePPNs
        printLog(
            f"Unique PPNs: {YELLOW}{len(uniquePPNs):,}{RESET} [{len(uniquePPNs) / len(df_per_year2[year]):.3%}]"
        )
        df_merge = df.copy()
        intersect = df_merge.merge(df_per_year2[year], on="PPN")
        printLog(
            f"Intersection with current records: {YELLOW}{len(intersect):,}{RESET} [{len(intersect) / len(df_merge):.3%}]"
        )

uniquePPNs = set(df["PPN"])
print("")
printLog("Current records:")
printLog(
    f"Unique PPNs: {YELLOW}{len(uniquePPNs):,}{RESET} [{len(uniquePPNs) / len(df):.3%}]"
)

[2026-11-03 20:40:36.027526]	Finished loading pickled records for year 2019.
[2026-11-03 20:40:36.614443]	Unique PPNs: 104,417 [87.014%]
[2026-11-03 20:40:36.665592]	Intersection with current records: 17,069 [14.224%]
[2026-11-03 20:40:37.088748]	Finished loading pickled records for year 2021.
[2026-11-03 20:40:37.948425]	Unique PPNs: 108,248 [90.207%]
[2026-11-03 20:40:37.984266]	Intersection with current records: 37,850 [31.542%]
[2026-11-03 20:40:38.409456]	Finished loading pickled records for year 2022.
[2026-11-03 20:40:38.909662]	Unique PPNs: 119,997 [99.997%]
[2026-11-03 20:40:38.948893]	Intersection with current records: 79,649 [66.374%]

[2026-11-03 20:40:38.984242]	Current records:
[2026-11-03 20:40:38.984561]	Unique PPNs: 119,988 [99.990%]


In [66]:
if loadMissingData:
    if not os.path.isfile(recordDatabaseFileLarge):
        saveFileFromURL(
            "https://8bit-inferno.de/_persistent/save_246k_dc_all_DEMO.pickle",
            ".",
        )
savedRecords = pickle.load(open(recordDatabaseFileLarge, "rb"))
printLog(f"Finished loading pickled records (large dataset).")

# as the values of the dict are lists, we have to make sure to the first element of each list to create a proper dataframe
r = [{key: values[0] for key, values in record.items()} for record in savedRecords]
df_large = pd.DataFrame(r)

[2026-11-03 20:40:42.399495]	Finished loading pickled records (large dataset).


Check wheter a dataset is a subset of the recently retrieved one.


In [101]:
diff_set_per_year = dict()
intersection_set_per_year = dict()
for year in df_per_year2:
    df_year_set = set(df_per_year2[year]["PPN"])
    diff_set_per_year[year] = df_year_set - df_large_set
    intersection_set_per_year[year] = df_large_set.intersection(df_year_set)
    print(f"{year} [df; unique len.: {len(df_set):,}]")
    print(
        f"\tis subset of 120k reference dataset : {df_year_set.issubset(df_set)};\n\tIntersection: {len(df_year_set.intersection(df_set)):,} elements."
    )
    print(f"{year} [df_large; unique len.: {len(df_large_set):,}]")
    print(
        f"\twith {len(df_year_set):,} elements is subset of full reference dataset: {df_year_set.issubset(df_large_set)};\n\tIntersection: {len(intersection_set_per_year[year]):,} elements;\n\tDifference: {len(diff_set_per_year[year]):,} elements."
    )

2019 [df; unique len.: 119,988]
	is subset of 120k reference dataset : False;
	Intersection: 13,251 elements.
2019 [df_large; unique len.: 246,504]
	with 104,417 elements is subset of full reference dataset: False;
	Intersection: 104,361 elements;
	Difference: 56 elements.
2021 [df; unique len.: 119,988]
	is subset of 120k reference dataset : False;
	Intersection: 31,642 elements.
2021 [df_large; unique len.: 246,504]
	with 108,248 elements is subset of full reference dataset: False;
	Intersection: 108,203 elements;
	Difference: 45 elements.
2022 [df; unique len.: 119,988]
	is subset of 120k reference dataset : False;
	Intersection: 79,637 elements.
2022 [df_large; unique len.: 246,504]
	with 119,997 elements is subset of full reference dataset: False;
	Intersection: 119,966 elements;
	Difference: 31 elements.


In [102]:
common = (
    set(df["PPN"])
    & set(df_per_year2[2022]["PPN"])
    & set(df_per_year2[2021]["PPN"])
    & set(df_per_year2[2019]["PPN"])
)
printLog(
    f"{YELLOW}{len(common):,}{RESET} [{len(common) / len(df):.3%}] common PPNs across all years; incl. duplicates."
)

[2026-11-04 20:59:55.373384]	9,710 [8.092%] common PPNs across all years; incl. duplicates.


In [50]:
result = df[df["PPN"].isin(common)]  # will include duplicates
result.describe()

,title,creator,subject,publisher,type,format,identifier,language,date,coverage,rights,PPN
count,9712,6900,9707,4384,9713,9713,9713,6653,8941,7737,8022,9713
unique,8963,3552,29,2370,8,1,9710,51,565,1104,5,9710
top,Tibetische Urkunde,"Weber, Gottfried",Historische Drucke,Decker,monograph,application/mets+xml,PPN3306131008,ger,1800,Leipzig,Public Domain Mark 1.0,PPN3306131008
freq,156,194,4776,69,5805,9713,2,2318,347,418,4742,2


# TODO

gibt es PPNs in alten Datensätzen, die heute nicht mehr existieren?


In [107]:
# large dataset
common = (
    set(df_large["PPN"])
    & set(df_per_year2[2022]["PPN"])
    & set(df_per_year2[2021]["PPN"])
    & set(df_per_year2[2019]["PPN"])
)
printLog(
    f"{YELLOW}{len(common):,}{RESET} [{len(common) / len(df_large):.3%}] common PPNs across all years; incl. duplicates."
)

result = df_large[df_large["PPN"].isin(common)]  # will include duplicates
result.describe()

[2026-11-04 21:01:37.184845]	29,462 [11.951%] common PPNs across all years; incl. duplicates.


,title,creator,subject,publisher,type,format,identifier,language,date,coverage,rights,PPN
count,29464,20826,29458,19439,29465,29465,29465,24285,28105,25720,25912,29465
unique,27663,11691,34,6441,8,1,29462,82,802,2486,5,29462
top,Tibetische Urkunde,"Schumann, Clara",Historische Drucke,Decker,monograph,application/mets+xml,PPN3306131008,ger,1915,Berlin,Public Domain Mark 1.0,PPN3306131008
freq,156,231,18077,311,20705,29465,2,14162,808,2719,21917,2


## A Short Note on Data Quality

The [DAMA model](https://kamal-ahmed.github.io/DQ-Dimensions.github.io/) contains dozens of data quality dimensions. However, in typical data quality assessment use cases only 5 to 20 dimensions are used.
The following 6 dimensions are the most relevant ones:

- **Accuracy**, i.e., how well the data reflect the real word; in other words: whether the data correctly describes the modeled properties
- **Completeness**, i.e., whether all needed data for a given use case is present
- **Consistency**, i.e., whether data is consistent in one ore more datasets, e.g. a German ZIP code has to have 5 digits or the date of birth of a person in different datasets must stay the same
- **Uniqueness/Deduplication**, i.e., if there are duplicate records describing the same real-world entity/event
- **Timeliness** i.e., how current the data is regarding the use case
- **Validity**, i.e., whether the data follows the expected syntax (format), rules, value ranges etc.

In the following steps, we will refer to these dimensions.

### Quality Control of the Extraction and Transformation

With two datasets at hand, we can check for syntactical differences between the recently retrieved and the old dataset. For instance, it is important whether the same Dublin Core fields are available as the following processing steps might rely on them.


In [58]:
if oldPickleAvailable:
    for year in availableYears:
        df_hist = df_per_year[year]
        l1 = len(list(df.columns))
        l2 = len(list(df_hist.columns))

        if l1 != l2:
            print(f"{UNDERLINE}{year}{RESET}")
            print(f"Warning! Number of Dublin Core fields differ between datasets.")
            sl1 = sorted(list(df.columns))
            sl2 = sorted(list(df_hist.columns))
            print(f"Current Dublin Core dataset contains %i fields." % l1)
            print(f"\t{sl1}")
            print(f"Historical Dublin Core dataset contains %i fields." % l2)
            print(f"\t{sl2}")
            print(
                f"Fields only available in old dataset of {year}: ",
                list(set(sl2) - set(sl1)),
            )
            print("Fields only available in new dataset: ", list(set(sl1) - set(sl2)))
            print("")
        else:
            print(
                f"{UNDERLINE}{year}{RESET} \nNo differences in the number of Dublin Core fields between current and historical dataset of year {year}."
            )
else:
    print(f"Comparison not available as file '{oldPickle}' is missing.")

2019
Warning! Number of Dublin Core fields differ between datasets.
Current Dublin Core dataset contains 12 fields.
	['PPN', 'coverage', 'creator', 'date', 'format', 'identifier', 'language', 'publisher', 'rights', 'subject', 'title', 'type']
Historical Dublin Core dataset contains 13 fields.
	['PPN', 'coverage', 'creator', 'date', 'format', 'identifier', 'language', 'publisher', 'relation', 'rights', 'subject', 'title', 'type']
Fields only available in old dataset of 2019:  ['relation']
Fields only available in new dataset:  []

2021
Warning! Number of Dublin Core fields differ between datasets.
Current Dublin Core dataset contains 12 fields.
	['PPN', 'coverage', 'creator', 'date', 'format', 'identifier', 'language', 'publisher', 'rights', 'subject', 'title', 'type']
Historical Dublin Core dataset contains 13 fields.
	['PPN', 'coverage', 'creator', 'date', 'format', 'identifier', 'language', 'publisher', 'relation', 'rights', 'subject', 'title', 'type']
Fields only available in old da